# Example of reading flux gain of [Liu et al. 2024](https://iopscience.iop.org/article/10.1088/1674-4527/ad5398)

In [1]:
import h5py

In [2]:
# Download the pre-measured flux gains from https://pan.cstcloud.cn/s/U6IEisNTQcg
fpath = 'flux_gain_Liu2024.hdf5'
nB = 1 # Beam number

## load the original flux gain

In [3]:
f = h5py.File(fpath, 'r')
print(f.keys())
# Ambient temperature (in degrees Celsius) of the pre-measured flux gains
ATemp_pre = f['Ambient_Temperature'][()]
# Zenith angle (in degrees) of the pre-measured flux gains
ZA_pre = f['ZA'][()]

print(f"Ambient_Temperature: {ATemp_pre} degree C")
print(f"Zenith angle: {ZA_pre} deg")

# frequency (in GHz) of the pre-measured flux gains
freq = f['freq'][()]
# flux gain in K/Jy
K_Jy = f[f'K_Jy{nB}'][()] 
print(K_Jy.shape)
K_Jy = K_Jy[0,:,0]

<KeysViewHDF5 ['Ambient_Temperature', 'K_Jy1', 'K_Jy10', 'K_Jy11', 'K_Jy12', 'K_Jy13', 'K_Jy14', 'K_Jy15', 'K_Jy16', 'K_Jy17', 'K_Jy18', 'K_Jy19', 'K_Jy2', 'K_Jy3', 'K_Jy4', 'K_Jy5', 'K_Jy6', 'K_Jy7', 'K_Jy8', 'K_Jy9', 'ZA', 'err_K_Jy1', 'err_K_Jy10', 'err_K_Jy11', 'err_K_Jy12', 'err_K_Jy13', 'err_K_Jy14', 'err_K_Jy15', 'err_K_Jy16', 'err_K_Jy17', 'err_K_Jy18', 'err_K_Jy19', 'err_K_Jy2', 'err_K_Jy3', 'err_K_Jy4', 'err_K_Jy5', 'err_K_Jy6', 'err_K_Jy7', 'err_K_Jy8', 'err_K_Jy9', 'freq']>
Ambient_Temperature: 15 degree C
Zenith angle: 10 deg
(1, 65536, 1)


## ambient temperature correction if needed

In [4]:
def ATemp_correction(ATemp_pre, ATemp_target, freq):
    """
    ATemp: Ambient temperature (in degrees Celsius) of the pre-measured flux gains
    ATemp_target: Ambient temperature (in degrees Celsius) of the target observation.
    freq: frequency (in MHz) of the pre-measured flux gains
    """
    s_factor = 4.7e-5*freq - 1.4e-3 # equation from Liu et al. 2024
    K_Jy_diff = s_factor * (ATemp_target - ATemp_pre)
    return K_Jy_diff

In [5]:
# Contact the FAST engineering team for the atmospheric/Ambient temperature data during your observations.
# Here weassume the Ambient temperature of the observation is 20 deg C as an example
ATemp_target = 20
K_Jy_target = K_Jy + ATemp_correction(ATemp_pre, ATemp_target, freq)
print(K_Jy_target)

[16.51284434 16.51293578 16.51302722 ... 15.5689395  15.56900233
 15.56906421]


## Zenith angle correction if needed

In [6]:
# assume the zenith angle of the observation is 5 deg
ZA_target = 5

- Load the ZA dependent from Jiang et al. 2019 by yourself

or

- Use hifast to load the ZA dependent from Jiang et al. 2019

In [7]:
from hifast.core.gain import gain_diff_from_ZA
import numpy as np
gain_diff = gain_diff_from_ZA(ZA_pre, np.array([ZA_target]), nB, freq)
gain_diff = gain_diff[0]
K_Jy_target_ZA_corr = K_Jy_target + gain_diff
print(K_Jy_target_ZA_corr)

[16.50146048 16.5015487  16.50163692 ... 15.43931276 15.4393752
 15.43943668]
